# Experimento de Controle de Dimensionalidade

## Motivação: o confundimento entre especificidade de domínio e maldição da dimensionalidade

Do Capítulo 3 para o Capítulo 4 da tese, **duas coisas mudaram ao mesmo tempo**:

| | Capítulo 3 | Capítulo 4 |
|---|---|---|
| **Representação textual** | Ollama genérico | FinBERT-PT-BR (domínio financeiro) |
| **Dimensionalidade** | 1.024 dims (+ PCA-32) | 5 dims |

O resultado reportado foi uma melhora de AUC de 0,610 (Ollama+PCA-32) para 0,670 (FinBERT 5-dim).  
Mas **qual fator causou a melhora?** A especificidade de domínio do FinBERT, ou simplesmente a redução drástica de dimensionalidade?

## Hipótese nula deste experimento

> *Se escolhermos 5 dimensões aleatórias dos embeddings Ollama (sem nenhum conhecimento de domínio financeiro), o desempenho será comparável ao FinBERT 5-dim.*

Caso a hipótese nula **não** seja rejeitada (i.e., AUC Ollama-5-random ≈ AUC FinBERT-5), a melhora do Capítulo 4 deve-se principalmente à redução de dimensionalidade, não à qualidade da representação.

Caso a hipótese nula **seja** rejeitada (AUC FinBERT-5 >> AUC Ollama-5-random), o FinBERT carrega sinal genuinamente superior para o problema.

## Protocolo

- **20 sorteios** de 5 colunas aleatórias do espaço Ollama (`emb_0` … `emb_1023`)
- Cada sorteio treina um **XGBoost** com a mesma configuração de todos os experimentos anteriores
- Split walk-forward 70/15/15, HORIZON=21, sem leakage de data
- AUC com IC bootstrap 95% (1.000 reamostragens) por sorteio
- Comparação contra: Ollama+PCA-32 = 0,610 | FinBERT 5-dim = 0,670 | 5-preço = 0,658

## 1. Imports e configuração

In [4]:
import sys, os
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from eval_utils import (
    walk_forward_split, evaluate_model, make_binary_target,
    format_metric_with_ci,
)

DATASET_PATH = '../2.stocks/dataset_full.csv'
HORIZON = 21
SEED = 42
N_SUBSETS = 20
SUBSET_SIZE = 5

# AUCs de referência das etapas anteriores
REF_OLLAMA_PCA32  = 0.610   # Cap. 3 — XGBoost + PCA-32 Ollama
REF_FINBERT_5DIM  = 0.670   # Cap. 4 — XGBoost + 5 FinBERT
REF_PRICE_5DIM    = 0.658   # Exp. 5.3 — XGBoost + 5 preço

np.random.seed(SEED)
print(f'N_SUBSETS={N_SUBSETS}, SUBSET_SIZE={SUBSET_SIZE}, HORIZON={HORIZON}, SEED={SEED}')

N_SUBSETS=20, SUBSET_SIZE=5, HORIZON=21, SEED=42


## 2. Carregamento do dataset e identificação das colunas de embedding

In [5]:
df = pd.read_csv(DATASET_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

# Construir target binário com horizonte de 21 dias
df['target'] = make_binary_target(df['Close'], horizon=HORIZON)
df = df.dropna(subset=['target']).reset_index(drop=True)

# Identificar colunas de embedding Ollama (padrão: 'emb_<número>')
EMB_COLS = [c for c in df.columns if c.startswith('emb_') and c[4:].isdigit()]
print(f'Dataset: {len(df)} linhas')
print(f'Colunas de embedding encontradas: {len(EMB_COLS)} (ex: {EMB_COLS[:5]} ... {EMB_COLS[-3:]})')
print(f'Balanço de classes: sobe={df["target"].mean():.3f}')

Dataset: 1206 linhas
Colunas de embedding encontradas: 1024 (ex: ['emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4'] ... ['emb_1021', 'emb_1022', 'emb_1023'])
Balanço de classes: sobe=0.590


## 3. Split walk-forward (único, reutilizado em todos os sorteios)

O split é feito uma única vez sobre o dataframe completo.  
O `StandardScaler` é refitado para cada subconjunto de colunas, mas **apenas sobre o treino** — sem leakage.

In [6]:
train_df, val_df, test_df = walk_forward_split(df, train_frac=0.70, val_frac=0.15)
print(f'Train: {len(train_df)} ({train_df["Date"].min().date()} → {train_df["Date"].max().date()})')
print(f'Val:   {len(val_df)} ({val_df["Date"].min().date()} → {val_df["Date"].max().date()})')
print(f'Test:  {len(test_df)} ({test_df["Date"].min().date()} → {test_df["Date"].max().date()})')

y_train = train_df['target'].values.astype(int)
y_val   = val_df['target'].values.astype(int)
y_test  = test_df['target'].values.astype(int)

pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos_weight = neg / max(pos, 1)
print(f'\nscale_pos_weight={scale_pos_weight:.3f} (pos={pos}, neg={neg})')

Train: 844 (2021-04-28 → 2024-09-10)
Val:   180 (2024-09-11 → 2025-06-04)
Test:  182 (2025-06-05 → 2026-02-25)

scale_pos_weight=0.755 (pos=481, neg=363)


## 4. Loop principal — 20 sorteios de 5 dimensões aleatórias

Para cada sorteio:
1. Selecionar 5 colunas `emb_*` sem reposição (semente diferente a cada iteração)
2. Normalizar com `StandardScaler` fitado no treino
3. Treinar XGBoost com configuração canônica (n_estimators=300, max_depth=4, lr=0.05)
4. Avaliar no conjunto de teste com IC bootstrap 95%

In [7]:
records = []

rng = np.random.default_rng(SEED)

for i in range(N_SUBSETS):
    # Sorteio de 5 colunas sem reposição
    subset_cols = list(rng.choice(EMB_COLS, size=SUBSET_SIZE, replace=False))

    # Normalização sem leakage
    scaler = StandardScaler().fit(train_df[subset_cols])
    X_train = scaler.transform(train_df[subset_cols])
    X_val   = scaler.transform(val_df[subset_cols])
    X_test  = scaler.transform(test_df[subset_cols])

    # XGBoost — mesma config de todos os experimentos anteriores
    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc',
        random_state=SEED,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    y_score = model.predict_proba(X_test)[:, 1]
    res = evaluate_model(y_test, y_score)

    records.append({
        'subset_id': i + 1,
        'cols': ','.join(subset_cols),
        'auc': res['auc'],
        'auc_lo': res['auc_ci'][0],
        'auc_hi': res['auc_ci'][1],
        'auc_str': res['auc_str'],
        'accuracy': res['accuracy'],
        'f1_pos': res['f1_pos'],
        'f1_neg': res['f1_neg'],
    })

    print(f'  Sorteio {i+1:2d}/20 | colunas: {subset_cols} | AUC: {res["auc_str"]}')

results_df = pd.DataFrame(records)
print('\nLoop concluído.')

  Sorteio  1/20 | colunas: [np.str_('emb_790'), np.str_('emb_448'), np.str_('emb_668'), np.str_('emb_91'), np.str_('emb_443')] | AUC: 0.518 [0.427, 0.610]
  Sorteio  2/20 | colunas: [np.str_('emb_997'), np.str_('emb_96'), np.str_('emb_537'), np.str_('emb_779'), np.str_('emb_752')] | AUC: 0.533 [0.442, 0.623]
  Sorteio  3/20 | colunas: [np.str_('emb_511'), np.str_('emb_856'), np.str_('emb_459'), np.str_('emb_379'), np.str_('emb_186')] | AUC: 0.592 [0.506, 0.672]
  Sorteio  4/20 | colunas: [np.str_('emb_556'), np.str_('emb_232'), np.str_('emb_460'), np.str_('emb_453'), np.str_('emb_839')] | AUC: 0.504 [0.410, 0.594]
  Sorteio  5/20 | colunas: [np.str_('emb_169'), np.str_('emb_875'), np.str_('emb_845'), np.str_('emb_282'), np.str_('emb_646')] | AUC: 0.479 [0.394, 0.572]
  Sorteio  6/20 | colunas: [np.str_('emb_912'), np.str_('emb_797'), np.str_('emb_455'), np.str_('emb_990'), np.str_('emb_693')] | AUC: 0.495 [0.398, 0.587]
  Sorteio  7/20 | colunas: [np.str_('emb_44'), np.str_('emb_507'),

## 5. Estatísticas de resumo dos 20 sorteios

In [8]:
auc_values = results_df['auc'].values

print('=== Ollama 5-dim aleatório — distribuição de AUC (20 sorteios) ===')
print(f'  Média:    {auc_values.mean():.4f}')
print(f'  Mediana:  {np.median(auc_values):.4f}')
print(f'  Std:      {auc_values.std():.4f}')
print(f'  Min:      {auc_values.min():.4f}  (sorteio #{results_df.loc[results_df["auc"].idxmin(), "subset_id"]})')
print(f'  Max:      {auc_values.max():.4f}  (sorteio #{results_df.loc[results_df["auc"].idxmax(), "subset_id"]})')
print(f'  Range:    {auc_values.min():.4f} – {auc_values.max():.4f}')
print()
print('=== Comparação com referências ===')
print(f'  Ollama+PCA-32  (Cap. 3):  {REF_OLLAMA_PCA32:.3f}')
print(f'  5-preço        (Exp 5.3): {REF_PRICE_5DIM:.3f}')
print(f'  FinBERT 5-dim  (Cap. 4):  {REF_FINBERT_5DIM:.3f}')
print(f'  Ollama 5-dim aleatório:   {auc_values.mean():.3f} ± {auc_values.std():.3f}  (média ± std)')
print()

delta_finbert = REF_FINBERT_5DIM - auc_values.mean()
pct_above_finbert = (auc_values >= REF_FINBERT_5DIM).mean() * 100
pct_above_price   = (auc_values >= REF_PRICE_5DIM).mean()   * 100

print(f'  Δ(FinBERT - Ollama-rand médio): {delta_finbert:+.4f}')
print(f'  Sorteios que superam FinBERT  : {pct_above_finbert:.0f}%')
print(f'  Sorteios que superam 5-preço  : {pct_above_price:.0f}%')

=== Ollama 5-dim aleatório — distribuição de AUC (20 sorteios) ===
  Média:    0.5085
  Mediana:  0.5137
  Std:      0.0566
  Min:      0.3628  (sorteio #18)
  Max:      0.6057  (sorteio #8)
  Range:    0.3628 – 0.6057

=== Comparação com referências ===
  Ollama+PCA-32  (Cap. 3):  0.610
  5-preço        (Exp 5.3): 0.658
  FinBERT 5-dim  (Cap. 4):  0.670
  Ollama 5-dim aleatório:   0.509 ± 0.057  (média ± std)

  Δ(FinBERT - Ollama-rand médio): +0.1615
  Sorteios que superam FinBERT  : 0%
  Sorteios que superam 5-preço  : 0%


## 6. Tabela completa dos sorteios

In [9]:
display_df = results_df[['subset_id', 'auc_str', 'auc', 'auc_lo', 'auc_hi', 'accuracy', 'f1_pos', 'f1_neg']].copy()
display_df = display_df.sort_values('auc', ascending=False).reset_index(drop=True)
display_df

,subset_id,auc_str,auc,auc_lo,auc_hi,accuracy,f1_pos,f1_neg
0,8,"0.606 [0.511, 0.694]",0.605726,0.511461,0.693935,0.631868,0.735178,0.396396
1,3,"0.592 [0.506, 0.672]",0.591837,0.506199,0.672171,0.626374,0.711864,0.468750
2,10,"0.573 [0.486, 0.665]",0.572562,0.486439,0.665412,0.565934,0.642534,0.447552
3,16,"0.564 [0.475, 0.655]",0.564342,0.475448,0.654583,0.587912,0.686192,0.400000
4,17,"0.551 [0.460, 0.644]",0.551446,0.459632,0.644016,0.560440,0.646018,0.420290
5,12,"0.534 [0.443, 0.629]",0.533872,0.443415,0.629318,0.549451,0.661157,0.327869
6,2,"0.533 [0.442, 0.623]",0.533022,0.442231,0.622988,0.587912,0.698795,0.347826
7,13,"0.526 [0.428, 0.610]",0.525794,0.428018,0.609513,0.532967,0.650206,0.297521
8,11,"0.524 [0.431, 0.611]",0.524235,0.431426,0.611379,0.587912,0.696356,0.358974
9,1,"0.518 [0.427, 0.610]",0.517999,0.426715,0.609828,0.571429,0.697674,0.264151


## 7. Salvar resultados

In [10]:
# Adicionar linha de resumo ao CSV
summary_row = pd.DataFrame([{
    'subset_id': 'MEAN',
    'cols': f'mean of {N_SUBSETS} random subsets of size {SUBSET_SIZE}',
    'auc': auc_values.mean(),
    'auc_lo': auc_values.mean() - auc_values.std(),
    'auc_hi': auc_values.mean() + auc_values.std(),
    'auc_str': f'{auc_values.mean():.3f} ±{auc_values.std():.3f}',
    'accuracy': results_df['accuracy'].mean(),
    'f1_pos': results_df['f1_pos'].mean(),
    'f1_neg': results_df['f1_neg'].mean(),
}])

output_df = pd.concat([results_df, summary_row], ignore_index=True)
output_path = 'results_dimensionality_control.csv'
output_df.to_csv(output_path, index=False)
print(f'Resultados salvos em: {output_path}')
print(f'Shape: {output_df.shape}')

Resultados salvos em: results_dimensionality_control.csv
Shape: (21, 9)


## 8. Interpretação

### O que este experimento responde

Este experimento isola o efeito da **dimensionalidade** do efeito da **qualidade da representação**.

Mantendo tudo igual (modelo XGBoost, split, horizonte, tamanho do vetor de features = 5), variamos apenas **quais 5 dimensões** usar — 5 aleatórias do Ollama vs. 5 do FinBERT.

### Cenários de interpretação

| Resultado observado | Interpretação | Implicação para a tese |
|---|---|---|
| AUC Ollama-rand médio << FinBERT (Δ > 0,05) | FinBERT captura sinal financeiro genuíno; a especificidade de domínio importa | A melhora Cap. 3 → Cap. 4 é atribuída principalmente à qualidade da representação |
| AUC Ollama-rand médio ≈ FinBERT (Δ < 0,02, ICs sobrepostos) | A maldição da dimensionalidade explica a maior parte da melhora | A tese deve enfatizar a redução dimensional como achado, não a especificidade do FinBERT |
| AUC Ollama-rand médio > FinBERT | Dimensões aleatórias superam o FinBERT | Resultado inesperado — pode indicar overfitting no pipeline do Cap. 4 |

### Nota sobre variância

Com apenas ~182 amostras de teste, a variância do estimador de AUC é alta. Um IC bootstrap que se sobrepõe a um valor de referência **não confirma** que os dois são iguais — apenas que a diferença não é estatisticamente distinguível neste tamanho amostral. A interpretação correta é sempre em conjunto com o IC.

### Próximos passos

- Se Ollama-rand ≈ FinBERT: adicionar subconjuntos de 5 dimensões **selecionados por correlação com o target** (como upper bound informado) para triangular o resultado.
- Se Ollama-rand << FinBERT: o experimento confirma o claim da tese — reportar o Δ como evidência da especificidade de domínio.